# Computer Simulation Analysis

In [47]:
msg = "Test this notebook3"
print(msg)

Test this notebook3


In [48]:
# .venv\Scripts\pip install -r requirements.txt

## Setup

In [49]:
import pandas as pd
import ast, re

## Helper Methods and Consts

In [56]:
DEP_VARS = [ 'revenue', 'cumulative-revenue', 'total-skill', 'hiring-threshold', 'coaching-rate', 'developers-leaving', 'count-turtles-on-patch']
DIST_STATS = ['mean', 'standard-deviation', 'median']

## Load CSV

In [57]:
df = pd.read_csv('output/test-experiment-1.csv', skiprows=34)

In [53]:
df.head(n=6)

,[all run data],[step],dependent-variables,revenue-distribution,skill-distribution,unemployment-rate
0,NaN,0,[[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 ...,[0 0 0],[4447.441077441077 1531.9509004964943 4408],0.000000
1,NaN,1,[[139257 127922 143819 122595 124013 132872 13...,[129324.11111111111 24981.203307927706 133151],[4446.939057239058 1531.8250277973143 4408.5],0.000000
2,NaN,2,[[139287 127952 143849 122625 124043 132902 13...,[128017.18181818182 28165.94944702413 132902],[4448.344444444445 1531.6910430001597 4409.5],0.000000
3,NaN,3,[[139317 127982 143879 122655 124073 132932 13...,[125168.12121212122 33453.35261386527 132050],[4448.340067340067 1531.3261918583294 4410],0.000000
4,NaN,4,[[139347 128012 143909 0 124103 132962 0 13561...,[121501.04040404041 39826.32904143587 131953],[4449.365779723813 1530.438735340006 4412],0.000337
5,NaN,5,[[139377 128042 143939 123040 124133 132992 13...,[132124.35353535353 16670.144718782918 133949],[4448.824520040418 1530.2187689964014 4411],0.000337


## Structure the data

In [61]:
def expand_col(df, col, names):
    df[[f'{col}.{n}' for n in names]] = df[col].apply(
        lambda x: pd.Series(ast.literal_eval(re.sub(r'\s+', ', ', str(x).strip()).replace('[, ', '[').replace(', ]', ']')))
    )

expand_col(df, 'dependent-variables', DEP_VARS)
expand_col(df, 'revenue-distribution', DIST_STATS)
expand_col(df, 'skill-distribution', DIST_STATS)

df[[c for c in df.columns if '.' in c]].head(n=2)

,dependent-variables.revenue,dependent-variables.cumulative-revenue,dependent-variables.total-skill,dependent-variables.hiring-threshold,dependent-variables.coaching-rate,dependent-variables.developers-leaving,dependent-variables.count-turtles-on-patch,revenue-distribution.mean,revenue-distribution.standard-deviation,revenue-distribution.median,skill-distribution.mean,skill-distribution.standard-deviation,skill-distribution.median
0,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[139227, 127892, 143789, 122565, 123983, 13284...","[2335, 1687, 1120, 1811, 2223, 1488, 2167, 134...","[7, 3, 0, 2, 1, 7, 6, 7, 8, 8, 1, 2, 9, 9, 4, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 3...",0.000000,0.000000,0.0,4447.441077,1531.950900,4408.0
1,"[139257, 127922, 143819, 122595, 124013, 13287...","[139257, 127922, 143819, 122595, 124013, 13287...","[139257, 127922, 143819, 122595, 124013, 13287...","[2335, 1687, 1120, 1811, 2223, 1488, 2167, 134...","[7, 3, 0, 2, 1, 7, 6, 7, 8, 8, 1, 2, 9, 9, 4, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 3...",129324.111111,24981.203308,133151.0,4446.939057,1531.825028,4408.5


In [ ]:
# Step 2

dep_cols    = [f'dependent-variables.{n}' for n in DEP_VARS]
scalar_cols = (
    ['[step]', 'unemployment-rate']
    + [f'revenue-distribution.{s}' for s in DIST_STATS]
    + [f'skill-distribution.{s}'   for s in DIST_STATS]
)

df_tidy = (
    df[scalar_cols + dep_cols]
    .explode(dep_cols) # one row per company
    .reset_index(drop=True)
)
df_tidy.insert(1, 'company_id', df_tidy.groupby('[step]').cumcount())
df_tidy = df_tidy.rename(columns={f'dependent-variables.{n}': n for n in DEP_VARS})

df_tidy.head(1000)


,[step],company_id,unemployment-rate,revenue-distribution.mean,revenue-distribution.standard-deviation,revenue-distribution.median,skill-distribution.mean,skill-distribution.standard-deviation,skill-distribution.median,revenue,cumulative-revenue,total-skill,hiring-threshold,coaching-rate,developers-leaving,count-turtles-on-patch
0,0,0,0.000000,0.000000,0.000000,0,4447.441077,1531.950900,4408.0,0,0,139227,2335,7,0,30
1,0,1,0.000000,0.000000,0.000000,0,4447.441077,1531.950900,4408.0,0,0,127892,1687,3,0,30
2,0,2,0.000000,0.000000,0.000000,0,4447.441077,1531.950900,4408.0,0,0,143789,1120,0,0,30
3,0,3,0.000000,0.000000,0.000000,0,4447.441077,1531.950900,4408.0,0,0,122565,1811,2,0,30
4,0,4,0.000000,0.000000,0.000000,0,4447.441077,1531.950900,4408.0,0,0,123983,2223,1,0,30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,10,5,0.000673,119882.585859,41467.249508,132133,4450.574461,1528.757534,4414.5,133142,1330070,133142,1488,7,0,30
996,10,6,0.000673,119882.585859,41467.249508,132133,4450.574461,1528.757534,4414.5,132566,1191006,132566,2167,6,0,30
997,10,7,0.000673,119882.585859,41467.249508,132133,4450.574461,1528.757534,4414.5,135792,1356570,135792,1344,7,0,30
998,10,8,0.000673,119882.585859,41467.249508,132133,4450.574461,1528.757534,4414.5,0,1102106,138311,2194,8,0,30


In [62]:
scalar_cols = (
    ['[step]', 'unemployment-rate']
    + [f'revenue-distribution.{s}' for s in DIST_STATS]
    + [f'skill-distribution.{s}'   for s in DIST_STATS]
)
dep_cols = [f'dependent-variables.{n}' for n in DEP_VARS]

df_company_step = (
    df[scalar_cols + dep_cols]
    .explode(dep_cols)
    .reset_index(drop=True)
    .assign(company_id=lambda d: d.groupby('[step]').cumcount())
    .rename(columns={f'dependent-variables.{n}': n for n in DEP_VARS})
    [['[step]', 'company_id'] + [c for c in scalar_cols[1:]] + DEP_VARS]
)

print(df_company_step.head(1000).to_string(index=False))

 [step]  company_id  unemployment-rate  revenue-distribution.mean  revenue-distribution.standard-deviation  revenue-distribution.median  skill-distribution.mean  skill-distribution.standard-deviation  skill-distribution.median revenue cumulative-revenue total-skill hiring-threshold coaching-rate developers-leaving count-turtles-on-patch
      0           0           0.000000                   0.000000                                 0.000000                          0.0              4447.441077                            1531.950900                     4408.0       0                  0      139227             2335             7                  0                     30
      0           1           0.000000                   0.000000                                 0.000000                          0.0              4447.441077                            1531.950900                     4408.0       0                  0      127892             1687             3                  0       